In [ ]:
import io
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn2

# Configuração global de estilo acadêmico
sns.set_theme(style="whitegrid")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={'width_ratios': [1, 1.35]})

# ==========================================
# PAINEL (A): Diagrama de Venn (Dados Atualizados)
# ==========================================
# Novos totais:
# Aave Only = 841,389 | Compound Only = 28,099 | Aave + Compound = 20,585
# Total Único = 890,073
aave_only = 841389
compound_only = 28099
overlap = 20585
total_wallets = aave_only + compound_only + overlap

v = venn2(
    subsets=(aave_only, compound_only, overlap),
    set_labels=('Aave Only', 'Compound Only'),
    ax=ax1
)

# Customização de cores e transparência
v.get_patch_by_id('10').set_color('#377eb8')
v.get_patch_by_id('10').set_alpha(0.7)

v.get_patch_by_id('01').set_color('#ff7f00')
v.get_patch_by_id('01').set_alpha(0.7)

v.get_patch_by_id('11').set_color('#2ca02c')
v.get_patch_by_id('11').set_alpha(0.9)

# 1. Ajuste explícito da posição do Overlap (subir o texto no eixo Y)
overlap_label = v.get_label_by_id('11')
if overlap_label:
    pct_overlap = (overlap / total_wallets) * 100
    overlap_label.set_text(f'{overlap:,}\n({pct_overlap:.2f}%)')
    overlap_label.set_fontsize(9.5)
    overlap_label.set_fontweight('bold')
    overlap_label.set_color('black')

    # Desloca ligeiramente para cima para não colidir com a borda
    pos_x, pos_y = overlap_label.get_position()
    overlap_label.set_position((pos_x, pos_y + 0.12))

# 2. Ajuste dos rótulos dos conjuntos exclusivos
label_a = v.get_label_by_id('10')
if label_a:
    pct_a = (aave_only / total_wallets) * 100
    label_a.set_text(f'{aave_only:,}\n({pct_a:.1f}%)')
    label_a.set_fontsize(9)

label_b = v.get_label_by_id('01')
if label_b:
    pct_b = (compound_only / total_wallets) * 100
    label_b.set_text(f'{compound_only:,}\n({pct_b:.1f}%)')
    label_b.set_fontsize(9)

ax1.set_title('(a) Aggregate Multi-Protocol Overlap', fontsize=11, fontweight='bold', pad=15)

# ==========================================
# PAINEL (B): Distribuição do Overlap por Chain
# ==========================================
csv_chain_data = """blockchain,combo,wallets
arbitrum,aave,183673
arbitrum,compound,3797
arbitrum,aave+compound,2662
avalanche_c,aave,81113
base,aave,209435
base,compound,9244
base,aave+compound,4767
bnb,aave,48658
celo,aave,3013
ethereum,aave,132739
ethereum,compound,18524
ethereum,aave+compound,6712
fantom,aave,1672
gnosis,aave,4426
linea,aave,13395
optimism,aave,103362
plasma,aave,4387
polygon,aave,224266
polygon,aave+compound,836
polygon,compound,664
scroll,aave,98943
sonic,aave,9027
unichain,compound,3674
zksync,aave,5247"""

df_chain = pd.read_csv(io.StringIO(csv_chain_data))
df_pivot = df_chain.pivot(index='blockchain', columns='combo', values='wallets').fillna(0)

# Filtrar apenas as redes onde existe o produto cruzado 'aave+compound'
df_overlap = df_pivot[df_pivot['aave+compound'] > 0].copy()

# Calcular penetração do overlap sobre o total de usuários do Compound na rede
df_overlap['total_compound'] = df_overlap['compound'] + df_overlap['aave+compound']
df_overlap['pct_compound'] = (df_overlap['aave+compound'] / df_overlap['total_compound']) * 100
df_overlap = df_overlap.sort_values(by='aave+compound', ascending=True)

# Plotagem do gráfico de barras horizontais
bars = ax2.barh(
    df_overlap.index.str.capitalize(),
    df_overlap['aave+compound'],
    color='#2ca02c',
    height=0.45
)

# Adicionar rótulos numéricos com volume absoluto e % do Compound
for bar, chain in zip(bars, df_overlap.index):
    val = bar.get_width()
    pct = df_overlap.loc[chain, 'pct_compound']
    ax2.text(
        val + 180,
        bar.get_y() + bar.get_height()/2.0,
        f'{int(val):,} ({pct:.1f}% of Compound users)',
        ha='left', va='center',
        fontsize=8.5, fontweight='bold', color='#333333'
    )

ax2.set_title('(b) Multi-Protocol Borrowers (Aave + Compound) by Chain', fontsize=11, fontweight='bold', pad=15)
ax2.set_xlabel('Number of Multi-Protocol Wallets', fontsize=10, fontweight='bold')
ax2.set_xlim(0, max(df_overlap['aave+compound']) * 1.55)

plt.suptitle('Q1.2 — Multi-Protocol Borrower Overlap Analysis', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()

# Salvar figuras
plt.savefig('Q1_2_overlap_panel.pdf', dpi=300, bbox_inches='tight')
plt.savefig('Q1_2_overlap_panel.png', dpi=300, bbox_inches='tight')
plt.show()